In [1]:
import os
import numpy as np
import pandas as pd

df = pd.read_csv("../data/raw/itsm_raw.csv", dtype=str, keep_default_na=False)
print(df.shape)

(46606, 25)


## Grouping the columns

In [3]:
%%writefile ../src/columns.py
id_cols       = ["Incident_ID", "KB_number", "Related_Interaction", "Related_Change"]
cat_cols      = ["CI_Name", "CI_Cat", "CI_Subcat", "WBS", "Category", "Closure_Code", "Status", "Alert_Status"]
ordinal_cols  = ["Impact", "Urgency", "Priority"]
count_cols    = ["No_of_Reassignments", "No_of_Related_Interactions", "No_of_Related_Incidents", "No_of_Related_Changes"]
time_cols     = ["Open_Time", "Reopen_Time", "Resolved_Time", "Close_Time"]
duration_cols = ["Handle_Time_hrs"]
drop_cols     = ["number_cnt"]



Overwriting ../src/columns.py


In [4]:
import sys, importlib
importlib.invalidate_caches()
sys.path.append("../src")
from columns import *

grouped = id_cols + cat_cols + ordinal_cols + count_cols + time_cols + duration_cols + drop_cols
print(len(grouped), set(df.columns) - set(grouped))

25 set()


In [5]:
print(open("../src/columns.py").read())

id_cols       = ["Incident_ID", "KB_number", "Related_Interaction", "Related_Change"]
cat_cols      = ["CI_Name", "CI_Cat", "CI_Subcat", "WBS", "Category", "Closure_Code", "Status", "Alert_Status"]
ordinal_cols  = ["Impact", "Urgency", "Priority"]
count_cols    = ["No_of_Reassignments", "No_of_Related_Interactions", "No_of_Related_Incidents", "No_of_Related_Changes"]
time_cols     = ["Open_Time", "Reopen_Time", "Resolved_Time", "Close_Time"]
duration_cols = ["Handle_Time_hrs"]
drop_cols     = ["number_cnt"]




## placeholders-to-nulls

In [6]:
placeholders = ["", "?", "NA", "N/A", "NaN", "nan", "null", "NULL", "None", "--", "-", "NS"]
for c in df.columns:
    df[c] = df[c].str.strip().replace(placeholders, np.nan)

df.isna().sum()[lambda s: s > 0].sort_values(ascending=False)

No_of_Related_Changes         46046
Related_Change                46046
No_of_Related_Incidents       45384
Reopen_Time                   44322
Resolved_Time                  1780
Impact                         1380
Priority                       1380
Closure_Code                    460
No_of_Related_Interactions      114
CI_Cat                          111
CI_Subcat                       111
No_of_Reassignments               1
Handle_Time_hrs                   1
dtype: int64

## flags, created before any type conversion

In [7]:
df["Has_Reopen_Time"] = df["Reopen_Time"].notna().astype(int)
df["Has_Related_Change"] = df["Related_Change"].notna().astype(int)
print(df[["Has_Reopen_Time", "Has_Related_Change"]].sum())

Has_Reopen_Time       2284
Has_Related_Change     560
dtype: int64


## fix types per group, with a self-check

In [8]:
num_cols = ordinal_cols + count_cols + duration_cols
before = df[num_cols].isna().sum()

df["Urgency"] = df["Urgency"].replace({"5 - Very Low": "5"})

for c in ordinal_cols + count_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df["Handle_Time_hrs"] = pd.to_numeric(df["Handle_Time_hrs"].str.replace(",", ".", regex=False), errors="coerce")

for c in time_cols:
    df[c] = pd.to_datetime(df[c], format="%d-%m-%Y %H:%M", errors="coerce")

print("New nulls created by conversion (empty means none):")
print((df[num_cols].isna().sum() - before)[lambda s: s != 0])
print(df[time_cols].isna().sum())
df.dtypes

New nulls created by conversion (empty means none):
Handle_Time_hrs    31252
dtype: int64
Open_Time            0
Reopen_Time      44322
Resolved_Time     1780
Close_Time           0
dtype: int64


CI_Name                                  str
CI_Cat                                   str
CI_Subcat                                str
WBS                                      str
Incident_ID                              str
Status                                   str
Impact                               float64
Urgency                                int64
Priority                             float64
number_cnt                               str
Category                                 str
KB_number                                str
Alert_Status                             str
No_of_Reassignments                  float64
Open_Time                     datetime64[us]
Reopen_Time                   datetime64[us]
Resolved_Time                 datetime64[us]
Close_Time                    datetime64[us]
Handle_Time_hrs                      float64
Closure_Code                             str
No_of_Related_Interactions           float64
Related_Interaction                      str
No_of_Rela

What matches expectations

* Null counts: Closure_Code is 460, as predicted. CI_Cat and CI_Subcat show 111 each, which means those are genuine blanks, not the word “Unknown”.
* Flags: Has_Reopen_Time is 2,284 and Has_Related_Change is 560.
* Datetimes: the null counts are 0, 44,322, 1,780 and 0, so the dd-mm-yyyy hh:mm format is correct for all four columns.
* Other conversions: Impact, Urgency, Priority, and the count columns converted with no new nulls.

In [9]:
raw = pd.read_csv("../data/raw/itsm_raw.csv", dtype=str, keep_default_na=False, usecols=["Handle_Time_hrs"])
h = raw["Handle_Time_hrs"].str.strip()
conv = pd.to_numeric(h.str.replace(",", ".", regex=False), errors="coerce")
bad = h[conv.isna() & (h != "")]

print("Failing values:", len(bad))
print(bad.str.replace(r"\d", "9", regex=True).value_counts().head(10))
print("Non-digit characters:", sorted({ch for v in bad for ch in v if not ch.isdigit()}))

Failing values: 31252
Handle_Time_hrs
9,99,99,99,999    30302
9,99,999            849
99,99,999            96
9,99,99,999           5
Name: count, dtype: int64
Non-digit characters: [',']


In [10]:
elapsed = (df["Close_Time"] - df["Open_Time"]).dt.total_seconds() / 3600
good = conv.notna()
bad_mask = conv.isna() & (h != "")
fixed = pd.to_numeric(h[bad_mask].str.replace(",", "", regex=False), errors="coerce") / 1e6

print("Decimals after comma in good rows:")
print(h[good & h.str.contains(",")].str.split(",").str[1].str.len().value_counts().head())

print("median Handle/elapsed, good rows :", (conv[good] / elapsed[good]).median())
print("median Handle/elapsed, fixed rows:", (fixed / elapsed[bad_mask]).median())

ex = bad[bad.str.replace(r"\d", "9", regex=True) == "9,999"].iloc[0]
print(repr(ex), [hex(ord(c)) for c in ex])

Decimals after comma in good rows:
Handle_Time_hrs
9    10909
3     2226
2     1150
4      689
1      123
Name: count, dtype: int64
median Handle/elapsed, good rows : 0.993055555
median Handle/elapsed, fixed rows: 50.92822272452809


IndexError: single positional indexer is out-of-bounds

In [11]:
lr = np.log10(conv[good] / elapsed[good]).replace([np.inf, -np.inf], np.nan).dropna()
print("Good rows, within 5% of elapsed:", round((lr.abs() < 0.021).mean(), 3))
print("Good rows, within a factor of 3:", round((lr.abs() < 0.477).mean(), 3))

d = pd.to_numeric(h[bad_mask].str.replace(",", "", regex=False), errors="coerce")
x = np.log10(d / elapsed[bad_mask]).replace([np.inf, -np.inf], np.nan).dropna()
frac = x - x.round()
print("Bad rows, digits match elapsed up to a power of 10:", round((frac.abs() < 0.021).mean(), 3))
print("Share expected by pure chance: about 0.042")
print("Implied powers of ten:")
print(x.round().astype(int).value_counts().head())

Good rows, within 5% of elapsed: 0.565
Good rows, within a factor of 3: 0.743
Bad rows, digits match elapsed up to a power of 10: 0.293
Share expected by pure chance: about 0.042
Implied powers of ten:
9    11151
8     9278
7     8158
6     1592
3      543
Name: count, dtype: int64


C:\Users\LAPTOPS24\anaconda3\envs\itsm\Lib\site-packages\pandas\core\arraylike.py:438: RuntimeWarning: divide by zero encountered in log10
  result = getattr(ufunc, method)(*inputs, **kwargs)


## The elapsed-time column

In [12]:
df["Elapsed_hrs"] = (df["Close_Time"] - df["Open_Time"]).dt.total_seconds() / 3600
print(df["Elapsed_hrs"].describe().round(1))
print("Close before Open:", (df["Elapsed_hrs"] < 0).sum())

count    46606.0
mean       123.1
std        452.8
min          0.0
25%          1.3
50%         18.4
75%         95.1
max      15722.6
Name: Elapsed_hrs, dtype: float64
Close before Open: 0


## null treatment per group

In [13]:
for c in ["No_of_Related_Incidents", "No_of_Related_Changes"]:
    df[c] = df[c].fillna(0)

df["Closure_Code"] = df["Closure_Code"].fillna("Not Recorded")
df["CI_Cat"] = df["CI_Cat"].fillna("Unknown")
df["CI_Subcat"] = df["CI_Subcat"].fillna("Unknown")

## Drop and save the checkpoint

In [14]:
df = df.drop(columns=["number_cnt", "Alert_Status"])
os.makedirs("../data/processed", exist_ok=True)
df.to_pickle("../data/processed/itsm_clean.pkl")
print(df.shape)
print(df.isna().sum()[lambda s: s > 0].sort_values(ascending=False))

(46606, 26)
Related_Change                46046
Reopen_Time                   44322
Handle_Time_hrs               31253
Resolved_Time                  1780
Impact                         1380
Priority                       1380
No_of_Related_Interactions      114
No_of_Reassignments               1
dtype: int64
